In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
import os
import re

# ==========================================
# 1. MOCK UTILITIES (Updated based on errors)
# ==========================================

class _NoisyTopKGate(nn.Module):
    def __init__(self, in_dim, num_experts, k, temperature, gate_input_dropout, gate_logits_dropout):
        super().__init__()
        self.k = k
        # FIX: Checkpoint contains biases, so we switch bias=True
        self.w_gate = nn.Linear(in_dim, num_experts, bias=True)
        self.w_noise = nn.Linear(in_dim, num_experts, bias=True)
        self.softplus = nn.Softplus()
        self.temperature = temperature
        
    def forward(self, h):
        logits = self.w_gate(h)
        probs = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        return probs, topk_idx

class _make_expert(nn.Module):
    def __init__(self, in_dim, hidden, num_classes, dropout=0.0):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        # FIX: The SoftMoE checkpoint has 4 layers (Linear, ReLU, Dropout, Linear)
        # So we ensure dropout layer is added if dropout > 0
        if dropout > 0: 
            layers.append(nn.Dropout(p=dropout))
        layers.append(nn.Linear(hidden, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, h): return self.net(h)


# ==========================================
# 2. MODEL DEFINITIONS
# ==========================================

class BaseHead(nn.Module):
    def pack(self, logits, probs, sel_idx, aux_loss, return_gate):
        output = {"logits": logits}
        if return_gate:
            output.update({"probs": probs, "sel_idx": sel_idx, "aux_loss": aux_loss})
            return output
        return logits

class DenseHead(BaseHead):
    head_name = "Dense"
    def __init__(self, in_dim, width, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, width),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.1), 
            nn.Linear(width, num_classes),
        )
    def forward(self, h, return_gate=False):
        return self.pack(self.fc(h), None, None, None, return_gate)

class SoftMoEHead(BaseHead):
    head_name = "SoftMoE"
    def __init__(self, in_dim, num_classes=10, num_experts=4, hidden_mult=0.5, temperature=1.0, dropout_p=0.0, **kwargs):
        super().__init__()
        self.num_experts = int(num_experts)
        self.temperature = float(temperature)
        self.gate = nn.Linear(in_dim, self.num_experts, bias=True)
        
        hidden = int(float(hidden_mult) * in_dim)
        # We pass dropout_p to ensure the structure matches (Linear -> ReLU -> Dropout -> Linear)
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(self.num_experts)])

    def forward(self, h, return_gate=False):
        gate_logits = self.gate(h)
        probs = F.softmax(gate_logits / self.temperature, dim=-1)
        expert_logits = torch.stack([expert(h) for expert in self.experts], dim=1)
        logits = (probs.unsqueeze(-1) * expert_logits).sum(dim=1)
        return self.pack(logits, probs, None, None, return_gate)

class SparseMoEHead(BaseHead):
    head_name = "SparseMoE"
    def __init__(self, in_dim, num_classes=10, num_experts=8, hidden_mult=0.0625, k=2, temperature=1.0, dropout_p=0.1, **kwargs):
        super().__init__()
        self.num_experts = int(num_experts)
        self.k = int(k)
        self.gate = _NoisyTopKGate(in_dim, self.num_experts, self.k, temperature, 0.0, 0.0)
        
        hidden = int(float(hidden_mult) * in_dim)
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(self.num_experts)])

    def forward(self, h, return_gate=True):
        probs, topk_idx = self.gate(h)
        return self.pack(h.new_zeros(h.shape[0], 10), probs, topk_idx, None, return_gate)

class MNISTFeatureBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        m = resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc = nn.Identity()
        self.backbone = m
        self.output_dim = 512
    def forward(self, x): return self.backbone(x)

class Classifier(nn.Module):
    def __init__(self, backbone, head): 
        super().__init__()
        self.backbone, self.head = backbone, head


# ==========================================
# 3. HELPER: PARSE CONFIG
# ==========================================
def get_model_config(path):
    config = {}
    config['num_experts'] = 8 
    config['k'] = 2
    
    x_match = re.search(r'X(\d+)', path)
    if x_match: config['num_experts'] = int(x_match.group(1))
        
    k_match = re.search(r'K(\d+)', path)
    if k_match: config['k'] = int(k_match.group(1))
        
    return config

# ==========================================
# 4. INSPECTION LOGIC
# ==========================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_CHECKPOINT_DIR = os.path.join(os.path.abspath('..'), 'checkpoints')

MODEL_PATHS = [
    'mnist/Dense/E50/model.pt',
    'mnist/SoftMoE/E50-X8/model.pt',
    'mnist/SparseMoE/E50-X8-K2/model.pt',
]

def inspect_model(relative_path):
    full_path = os.path.join(BASE_CHECKPOINT_DIR, relative_path)
    print("\n" + "="*60)
    print(f"📂 PROCESSING: {relative_path}")
    print("="*60)

    if not os.path.exists(full_path):
        print("❌ File not found. Skipping...")
        return

    # 1. Config & Instantiation
    backbone = MNISTFeatureBackbone()
    in_dim = 512
    num_classes = 10
    cfg = get_model_config(relative_path)
    
    if "Dense" in relative_path:
        head = DenseHead(in_dim, width=512, num_classes=num_classes)
        
    elif "SoftMoE" in relative_path:
        print(f"   Detected SoftMoE (Experts={cfg['num_experts']})")
        # FIX: hidden_mult=0.125 (to get hidden=64)
        # FIX: dropout_p=0.1 (to force the extra Dropout layer to match checkpoint structure)
        head = SoftMoEHead(in_dim, num_experts=cfg['num_experts'], hidden_mult=0.125, dropout_p=0.1)
        
    elif "SparseMoE" in relative_path:
        print(f"   Detected SparseMoE (Experts={cfg['num_experts']}, K={cfg['k']})")
        # FIX: hidden_mult=0.125 (to get hidden=64)
        head = SparseMoEHead(in_dim, num_experts=cfg['num_experts'], k=cfg['k'], hidden_mult=0.125)
        
    else:
        print("❓ Unknown model type. Skipping.")
        return

    model = Classifier(backbone, head).to(DEVICE)

    # 2. Load
    try:
        checkpoint = torch.load(full_path, map_location=DEVICE)
        
        if isinstance(checkpoint, dict):
            if 'model' in checkpoint: state_dict = checkpoint['model']
            elif 'model_state_dict' in checkpoint: state_dict = checkpoint['model_state_dict']
            else: state_dict = checkpoint
            torch.save(checkpoint['model'])
        else:
            state_dict = checkpoint

        model.load_state_dict(state_dict)
        print("✅ Weights loaded successfully!")
        
    except RuntimeError as e:
        print(f"❌ SIZE MISMATCH:\n{e}")
        return
    except Exception as e:
        print(f"❌ Error: {e}")
        return

    # 3. Stats
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Total Parameters: {total_params:,}")


# ==========================================
# 5. RUN
# ==========================================
print(f"Found {len(MODEL_PATHS)} models to process...\n")
for path in MODEL_PATHS:
    inspect_model(path)

Found 3 models to process...


📂 PROCESSING: mnist/Dense/E50/model.pt
✅ Weights loaded successfully!
📊 Total Parameters: 11,435,466

📂 PROCESSING: mnist/SoftMoE/E50-X8/model.pt
   Detected SoftMoE (Experts=8)
✅ Weights loaded successfully!
📊 Total Parameters: 11,439,640

📂 PROCESSING: mnist/SparseMoE/E50-X8-K2/model.pt
   Detected SparseMoE (Experts=8, K=2)
✅ Weights loaded successfully!
📊 Total Parameters: 11,443,744


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
import os
import re

# ==========================================
# 1. MOCK UTILITIES & LAYERS
# ==========================================

class _NoisyTopKGate(nn.Module):
    """Mock Gate to match saved weights (bias=True based on error logs)"""
    def __init__(self, in_dim, num_experts, k, temperature, gate_input_dropout, gate_logits_dropout):
        super().__init__()
        self.k = k
        self.w_gate = nn.Linear(in_dim, num_experts, bias=True)
        self.w_noise = nn.Linear(in_dim, num_experts, bias=True)
        self.softplus = nn.Softplus()
        self.temperature = temperature
        
    def forward(self, h):
        logits = self.w_gate(h)
        probs = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        return probs, topk_idx

class _make_expert(nn.Module):
    """Standard MLP Expert with optional dropout to match architecture"""
    def __init__(self, in_dim, hidden, num_classes, dropout=0.0):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        # Fix: Ensure dropout layer exists if p > 0 to match layer count in checkpoint
        if dropout > 0: 
            layers.append(nn.Dropout(p=dropout))
        layers.append(nn.Linear(hidden, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, h): return self.net(h)


# ==========================================
# 2. MODEL ARCHITECTURES
# ==========================================

class BaseHead(nn.Module):
    def pack(self, logits, probs, sel_idx, aux_loss, return_gate):
        output = {"logits": logits}
        if return_gate:
            output.update({"probs": probs, "sel_idx": sel_idx, "aux_loss": aux_loss})
            return output
        return logits

class DenseHead(BaseHead):
    head_name = "Dense"
    def __init__(self, in_dim, width, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, width),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.1), 
            nn.Linear(width, num_classes),
        )
    def forward(self, h, return_gate=False):
        return self.pack(self.fc(h), None, None, None, return_gate)

class SoftMoEHead(BaseHead):
    head_name = "SoftMoE"
    def __init__(self, in_dim, num_classes=10, num_experts=4, hidden_mult=0.5, temperature=1.0, dropout_p=0.0, **kwargs):
        super().__init__()
        self.num_experts = int(num_experts)
        self.temperature = float(temperature)
        self.gate = nn.Linear(in_dim, self.num_experts, bias=True)
        
        hidden = int(float(hidden_mult) * in_dim)
        # Fix: Explicitly use dropout_p to ensure 4-layer structure
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(self.num_experts)])

    def forward(self, h, return_gate=False):
        gate_logits = self.gate(h)
        probs = F.softmax(gate_logits / self.temperature, dim=-1)
        expert_logits = torch.stack([expert(h) for expert in self.experts], dim=1)
        logits = (probs.unsqueeze(-1) * expert_logits).sum(dim=1)
        return self.pack(logits, probs, None, None, return_gate)

class SparseMoEHead(BaseHead):
    head_name = "SparseMoE"
    def __init__(self, in_dim, num_classes=10, num_experts=8, hidden_mult=0.0625, k=2, temperature=1.0, dropout_p=0.1, **kwargs):
        super().__init__()
        self.num_experts = int(num_experts)
        self.k = int(k)
        self.gate = _NoisyTopKGate(in_dim, self.num_experts, self.k, temperature, 0.0, 0.0)
        
        hidden = int(float(hidden_mult) * in_dim)
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(self.num_experts)])

    def forward(self, h, return_gate=True):
        probs, topk_idx = self.gate(h)
        return self.pack(h.new_zeros(h.shape[0], 10), probs, topk_idx, None, return_gate)

class MNISTFeatureBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        m = resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc = nn.Identity()
        self.backbone = m
        self.output_dim = 512
    def forward(self, x): return self.backbone(x)

class Classifier(nn.Module):
    def __init__(self, backbone, head): 
        super().__init__()
        self.backbone, self.head = backbone, head


# ==========================================
# 3. ANALYSIS TOOLS
# ==========================================

def analyze_model(model, name="Model"):
    """Deep dive into weights, experts, and gating"""
    print(f"   🔎 ANALYZING WEIGHTS...")
    
    # A. Global Stats
    all_params = torch.cat([p.detach().view(-1) for p in model.parameters()])
    print(f"      Global Mean: {all_params.mean().item():.5f} | Std: {all_params.std().item():.5f}")
    
    # B. Expert Diversity (The "MoE Test")
    if hasattr(model.head, 'experts') and len(model.head.experts) > 1:
        e0 = model.head.experts[0].net[0].weight
        e1 = model.head.experts[1].net[0].weight
        diff = (e0 - e1).abs().mean().item()
        
        print(f"      🧠 Expert Diversity (E0 vs E1 diff): {diff:.6f}")
        if diff < 1e-7:
            print("         ⚠️ WARNING: Experts are identical (Collapse).")
        else:
            print("         ✅ Experts are specialized.")

    # C. Gate Sharpness
    if hasattr(model.head, 'gate'):
        if hasattr(model.head.gate, 'w_gate'): # SparseMoE
            gate_w = model.head.gate.w_gate.weight
        else: # SoftMoE
            gate_w = model.head.gate.weight
        print(f"      🚪 Gate Max Weight: {gate_w.max().item():.5f} (Higher = Sharper routing)")

from torchinfo import summary

def show_model_structure(model):
    print("\n" + "="*40)
    print("🏗️ MODEL ARCHITECTURE & SIZES")
    print("="*40)
    
    # Input size: (Batch_Size, Channels, Height, Width)
    # MNIST is (1, 1, 28, 28)
    summary(model, input_size=(1, 1, 28, 28), 
            col_names=["input_size", "output_size", "num_params"],
            depth=4) # Depth=4 lets you see inside the "Experts"


# ==========================================
# 4. MAIN EXECUTION
# ==========================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_CHECKPOINT_DIR = os.path.join(os.path.abspath('..'), 'checkpoints')

MODEL_PATHS = [
    'mnist/Dense/E50/model.pt',
    'mnist/SoftMoE/E50-X8/model.pt',
    'mnist/SparseMoE/E50-X8-K2/model.pt',
]

def get_model_config(path):
    config = {'num_experts': 8, 'k': 2}
    x_match = re.search(r'X(\d+)', path)
    if x_match: config['num_experts'] = int(x_match.group(1))
    k_match = re.search(r'K(\d+)', path)
    if k_match: config['k'] = int(k_match.group(1))
    return config

print(f"Processing {len(MODEL_PATHS)} models from {BASE_CHECKPOINT_DIR}...\n")

for path in MODEL_PATHS:
    full_path = os.path.join(BASE_CHECKPOINT_DIR, path)
    print("="*60)
    print(f"📂 FILE: {path}")

    if not os.path.exists(full_path):
        print("❌ File not found.")
        continue

    # 1. SETUP ARCHITECTURE
    backbone = MNISTFeatureBackbone()
    cfg = get_model_config(path)
    
    if "Dense" in path:
        head = DenseHead(512, width=512, num_classes=10)
    elif "SoftMoE" in path:
        head = SoftMoEHead(512, num_experts=cfg['num_experts'], hidden_mult=0.125, dropout_p=0.1)
    elif "SparseMoE" in path:
        head = SparseMoEHead(512, num_experts=cfg['num_experts'], k=cfg['k'], hidden_mult=0.125)
    else:
        print("❓ Unknown type.")
        continue

    model = Classifier(backbone, head).to(DEVICE)

    # 2. LOAD WEIGHTS
    try:
        checkpoint = torch.load(full_path, map_location=DEVICE)
        state_dict = checkpoint.get('model', checkpoint.get('model_state_dict', checkpoint))
        model.load_state_dict(state_dict)
        print("✅ Weights Loaded")
        
        # 3. PRINT STRUCTURE (Isolated Try/Catch)
        try:
            # FIX: Pass 'device=DEVICE' to ensure dummy input matches model location
            #summary(model, input_size=(1, 1, 28, 28), 
            #        col_names=["input_size", "output_size", "num_params"], 
                    depth=4, 
                    device=DEVICE)
        except Exception as e:
            print(f"⚠️ Torchinfo failed ({e}). using native print:")
            print(model) # Fallback to standard print

        # 4. ANALYZE (Run this even if torchinfo fails)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"\n📊 Size: {total_params:,} params ({total_params*4/1024**2:.1f} MB)")
        analyze_model(model, path)
        
    except RuntimeError as e:
        print(f"❌ FATAL LOAD ERROR: {e}")
    except Exception as e:
        print(f"❌ GENERAL ERROR: {e}")

Processing 3 models from /home/dani/sem_3/ds_project/mixture-of-experts-project/checkpoints...

📂 FILE: mnist/Dense/E50/model.pt
✅ Weights Loaded
⚠️ Torchinfo failed (Failed to run torchinfo. See above stack traces for more details. Executed layers up to: []). using native print:
Classifier(
  (backbone): MNISTFeatureBackbone(
    (backbone): ResNet(
      (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): Identity()
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): Bat

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
import os
import re

# --- 1. Model Definitions ---

class _NoisyTopKGate(nn.Module):
    def __init__(self, in_dim, num_experts, k, temperature, gate_input_dropout, gate_logits_dropout):
        super().__init__()
        self.k = k
        self.w_gate = nn.Linear(in_dim, num_experts, bias=True)
        self.w_noise = nn.Linear(in_dim, num_experts, bias=True)
        self.softplus = nn.Softplus()
        self.temperature = temperature
    def forward(self, h):
        logits = self.w_gate(h)
        probs = F.softmax(logits, dim=-1)
        _, topk_idx = torch.topk(probs, self.k, dim=-1)
        return probs, topk_idx

class _make_expert(nn.Module):
    def __init__(self, in_dim, hidden, num_classes, dropout=0.0):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        if dropout > 0: layers.append(nn.Dropout(p=dropout))
        layers.append(nn.Linear(hidden, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, h): return self.net(h)

class BaseHead(nn.Module):
    def pack(self, logits, probs, sel_idx, aux_loss, return_gate):
        # For Netron/ONNX, return only logits to keep the graph simple
        if torch.onnx.is_in_onnx_export():
            return logits
        return {"logits": logits}

class DenseHead(BaseHead):
    def __init__(self, in_dim, width, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, width), nn.ReLU(inplace=True),
            nn.Dropout(p=0.1), nn.Linear(width, num_classes)
        )
    def forward(self, h, return_gate=False):
        return self.pack(self.fc(h), None, None, None, return_gate)

class SoftMoEHead(BaseHead):
    def __init__(self, in_dim, num_classes=10, num_experts=4, hidden_mult=0.5, temperature=1.0, dropout_p=0.0):
        super().__init__()
        hidden = int(float(hidden_mult) * in_dim)
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(int(num_experts))])
        self.gate = nn.Linear(in_dim, int(num_experts), bias=True)
        self.temperature = temperature
    def forward(self, h, return_gate=False):
        probs = F.softmax(self.gate(h) / self.temperature, dim=-1)
        expert_logits = torch.stack([e(h) for e in self.experts], dim=1)
        logits = (probs.unsqueeze(-1) * expert_logits).sum(dim=1)
        return self.pack(logits, probs, None, None, return_gate)

class SparseMoEHead(BaseHead):
    def __init__(self, in_dim, num_classes=10, num_experts=8, hidden_mult=0.0625, k=2, temperature=1.0, dropout_p=0.1):
        super().__init__()
        hidden = int(float(hidden_mult) * in_dim)
        self.experts = nn.ModuleList([_make_expert(in_dim, hidden, num_classes, dropout_p) for _ in range(int(num_experts))])
        self.gate = _NoisyTopKGate(in_dim, int(num_experts), int(k), temperature, 0.0, 0.0)
    def forward(self, h, return_gate=True):
        probs, topk_idx = self.gate(h)
        # CRITICAL FOR ONNX: Force data flow through experts so Netron sees them
        expert_outputs = torch.stack([e(h) for e in self.experts], dim=1)
        logits = (probs.unsqueeze(-1) * expert_outputs).sum(dim=1)
        return self.pack(logits, probs, topk_idx, None, return_gate)

class MNISTFeatureBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()
    def forward(self, x): return self.backbone(x)

class Classifier(nn.Module):
    def __init__(self, backbone, head): 
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, x):
        return self.head(torch.flatten(self.backbone(x), 1))

# --- 2. Configuration & Paths ---

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = os.path.join(os.path.abspath('..'), 'checkpoints')
PATHS = [
    'mnist/Dense/E50/model.pt',
    'mnist/SoftMoE/E50-X8/model.pt',
    'mnist/SparseMoE/E50-X8-K2/model.pt',
]

def get_config(path):
    c = {'experts': 8, 'k': 2}
    if 'X' in path: c['experts'] = int(re.search(r'X(\d+)', path).group(1))
    if 'K' in path: c['k'] = int(re.search(r'K(\d+)', path).group(1))
    return c

# --- 3. Export Loop ---

for rel_path in PATHS:
    full_path = os.path.join(BASE_DIR, rel_path)
    if not os.path.exists(full_path):
        print(f"Skipping {rel_path} (not found)")
        continue

    # Instantiate
    backbone = MNISTFeatureBackbone()
    cfg = get_config(rel_path)
    if "Dense" in rel_path:
        head = DenseHead(512, 512, 10)
    elif "SoftMoE" in rel_path:
        head = SoftMoEHead(512, num_experts=cfg['experts'], hidden_mult=0.125, dropout_p=0.1)
    else: # SparseMoE
        head = SparseMoEHead(512, num_experts=cfg['experts'], k=cfg['k'], hidden_mult=0.125)

    model = Classifier(backbone, head).to(DEVICE)
    
    # Load Weights
    ckpt = torch.load(full_path, map_location=DEVICE)
    state = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()

    # Export
    onnx_path = full_path.replace('.pt', '.onnx')
    dummy_input = torch.randn(1, 1, 28, 28, device=DEVICE)
    try:
        torch.onnx.export(model, dummy_input, onnx_path, 
                          input_names=['input'], output_names=['logits'], 
                          opset_version=18)
        print(f"Saved: {onnx_path}")
    except Exception as e:
        print(f"Failed {rel_path}: {e}")

[torch.onnx] Obtain model graph for `Classifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Classifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 40 of general pattern rewrite rules.
Saved: /home/dani/sem_3/ds_project/mixture-of-experts-project/checkpoints/mnist/Dense/E50/model.onnx
[torch.onnx] Obtain model graph for `Classifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Classifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
Saved: /home/dani/sem_3/ds_project/mixture-of-experts-project